# Data Loading & Standardization

In [ ]:
import pandas as pd
import numpy as np
import re
import os
from rapidfuzz import process, utils
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
import optuna

# Global styling for high-quality visuals
sns.set_theme(style='whitegrid', context='notebook', palette='muted')

# File Paths for RBI datasets
file_3_5 = 'Table No 3.5 Population Group and Bank Group-wise Classification of Outstanding Credit of SCBs According to Occupation.xlsx'
file_restructuring = '13.Loan Subjected to Restructuring and Corporate Debt Restructured.xlsx'
file_npa = '_6.Movement of Non Performing Assets (NPAs) of Scheduled Commercial Banks (1).xlsx'

# Load Raw Data
table_3_5 = pd.read_excel(file_3_5)
table_restructuring = pd.read_excel(file_restructuring)
table_npa = pd.read_excel(file_npa)

def to_camel_case(text):
    if pd.isna(text) or str(text).strip() == "": return "unnamedColumn"
    text = str(text)
    words = re.findall(r'[A-Z]?[a-z0-9]+|[A-Z]+(?=[A-Z][a-z0-9]|\b)', text)
    if not words: words = re.sub(r'[^a-zA-Z0-9]', ' ', text).split()
    if not words: return "unnamedColumn"
    processed = [words[0].lower()]
    for word in words[1:]: processed.append(word.capitalize())
    return "".join(processed)

print("Data loaded successfully.")

# Initial Inspection

In [ ]:
def inspect_table(df, name):
    print(f"--- Inspection: {name} ---")
    print(f"Shape: {df.shape}, Columns: {df.columns.tolist()[:5]}...")
    display(df.head(3))

inspect_table(table_3_5, 'Table 3.5')
inspect_table(table_restructuring, 'Restructuring')
inspect_table(table_npa, 'NPA Movement')

# First Column Validation & Fix

In [ ]:
def validate_first_column(df):
    if df.empty: return df
    first_col = df.columns[0]
    if df[first_col].isna().all():
        df = df.drop(columns=[first_col])
    elif pd.api.types.is_numeric_dtype(df[first_col]):
        df[first_col] = df[first_col].fillna(df[first_col].mean())
    return df

table_3_5 = validate_first_column(table_3_5)
table_restructuring = validate_first_column(table_restructuring)
table_npa = validate_first_column(table_npa)
print('First column logic applied.')

# Unnamed Column Renaming

In [ ]:
def infer_column_names(df):
    new_columns = list(df.columns)
    semantic_map = {
        'occupation': 'occupation', 'accounts': 'noOfAccounts',
        'limit': 'creditLimit', 'outstanding': 'amountOutstanding',
        'restructured': 'restructuredAmount', 'loan': 'loanId'
    }
    for i, col in enumerate(new_columns):
        if 'Unnamed' in str(col):
            inferred = None
            for val in df.iloc[:15, i]:
                val_str = str(val).lower()
                for key, mapped in semantic_map.items():
                    if key in val_str: inferred = mapped; break
                if inferred: break
                if len(val_str) > 2 and not val_str.replace('.','').isdigit():
                    inferred = val_str.strip(); break
            if inferred: new_columns[i] = inferred
    df.columns = new_columns
    return df

table_3_5 = infer_column_names(table_3_5)
table_restructuring = infer_column_names(table_restructuring)
table_npa = infer_column_names(table_npa)
print('Unnamed columns handled semantically.')

# Row-Level Cleaning

In [ ]:
def clean_row_levels(df):
    if 1 in df.index:
        row_1 = pd.Series(df.loc[1]).ffill()
        new_cols = list(df.columns)
        for i, val in enumerate(row_1):
            if pd.notna(val) and str(val).strip() != '' and ('Unnamed' in str(new_cols[i]) or 'unnamed' in str(new_cols[i]).lower()):
                new_cols[i] = str(val).strip()
        df.columns = new_cols
    if 2 in df.index:
        df = df.drop(index=2)
    return df

table_3_5 = clean_row_levels(table_3_5)
table_restructuring = clean_row_levels(table_restructuring)
table_npa = clean_row_levels(table_npa)
print('Row-level cleaning complete.')

# Column Name Standardization

In [ ]:
def standardize_columns(df):
    df.columns = [to_camel_case(col) for col in df.columns]
    new_cols, counts = [], {}
    for col in df.columns:
        if col in counts:
            counts[col] += 1
            new_cols.append(f'{col}_{counts[col]}')
        else:
            counts[col] = 0
            new_cols.append(col)
    df.columns = new_cols
    return df

table_3_5 = standardize_columns(table_3_5)
table_restructuring = standardize_columns(table_restructuring)
table_npa = standardize_columns(table_npa)
print('Columns standardized.')

# Final Cleaned Output

In [ ]:
# Sub-Process 1.1: Temporal Normalization
def extract_fiscal_year(val):
    if pd.isna(val): return None
    s = str(val).strip()
    matches = re.findall(r'20(\d{2})', s)
    return int("20" + matches[-1]) if matches else None

def apply_temporal(df, name):
    year_col = next((col for col in df.columns if any(x in col.lower() for x in ['year', 'march', 'unnamedColumn'])), df.columns[0])
    df['fiscalYear'] = df[year_col].apply(extract_fiscal_year).ffill()
    df = df[df['fiscalYear'] >= 2018].copy()
    print(f"Unique fiscalYears for {name}: {sorted(df['fiscalYear'].unique())}")
    return df

table_3_5 = apply_temporal(table_3_5, "Table 3.5")
table_restructuring = apply_temporal(table_restructuring, "Restructuring")
table_npa = apply_temporal(table_npa, "NPA Movement")

# Sub-Process 1.2: Entity Resolution
def resolve_banks(df, master_list):
    df = df.copy()
    bank_col = next((col for col in df.columns if 'bank' in col.lower() and col != 'bankName'), df.columns[1])
    df['rawBankName'] = df[bank_col].map(lambda x: re.sub(r'[^A-Z0-9 ]', '', str(x).upper()).strip())
    mapping = {n: process.extractOne(n, master_list, processor=utils.default_process)[0] 
               if len(n) > 3 and process.extractOne(n, master_list, processor=utils.default_process)[1] > 80 
               else n for n in df['rawBankName'].unique() if n}
    df['bankName'] = df['rawBankName'].map(mapping)
    return df

table_npa['bankName'] = table_npa.iloc[:, 1].map(lambda x: re.sub(r'[^A-Z0-9 ]', '', str(x).upper()).strip())
master_bank_list = [b for b in table_npa['bankName'].unique() if len(b) > 3]
table_restructuring = resolve_banks(table_restructuring, master_bank_list)

# Sub-Process 1.3: Sectoral Aggregation
def aggregate_3_5_final(df):
    limit_cols = [c for c in df.columns if 'limit' in c.lower()]
    out_cols = [c for c in df.columns if 'outstanding' in c.lower()]
    occ_col = next((col for col in df.columns if 'occupation' in col.lower()), df.columns[0])
    df_l = pd.melt(df, id_vars=['fiscalYear', occ_col], value_vars=limit_cols, var_name='attr', value_name='limit')
    df_l['bankGroup'] = df_l['attr'].apply(lambda x: 'Public' if 'public' in x.lower() else ('Private' if 'private' in x.lower() else 'Foreign'))
    feat_l = df_l.groupby(['fiscalYear', 'bankGroup']).agg({'limit': 'sum'}).reset_index().rename(columns={'limit': 'totalCreditLimit'})
    df_o = pd.melt(df, id_vars=['fiscalYear', occ_col], value_vars=out_cols, var_name='attr', value_name='out')
    df_o['bankGroup'] = df_o['attr'].apply(lambda x: 'Public' if 'public' in x.lower() else ('Private' if 'private' in x.lower() else 'Foreign'))
    feat_sector = df_o.pivot_table(index=['fiscalYear', 'bankGroup'], columns=occ_col, values='out', aggfunc='sum').reset_index()
    feat_sector.columns = [to_camel_case(f"credit_{c}") if c not in ['fiscalYear', 'bankGroup'] else c for c in feat_sector.columns]
    feat_total_out = df_o.groupby(['fiscalYear', 'bankGroup']).agg({'out': 'sum'}).reset_index().rename(columns={'out': 'totalAdvances'})
    merged = pd.merge(feat_l, feat_total_out, on=['fiscalYear', 'bankGroup'])
    return pd.merge(merged, feat_sector, on=['fiscalYear', 'bankGroup'])

df_3_5_features = aggregate_3_5_final(table_3_5)

# Sub-Process 1.4: Merging
def assign_group(name):
    if any(x in str(name).upper() for x in ['STATE BANK', 'CANARA', 'PUNJAB', 'INDIAN', 'BARODA', 'CENTRAL']): return 'Public'
    return 'Private'

table_npa['bankGroup'] = table_npa['bankName'].apply(assign_group)
npa_num_cols = table_npa.select_dtypes(include=[np.number]).columns
table_npa['npaClosingBalance'] = pd.to_numeric(table_npa[npa_num_cols[-1]], errors='coerce').fillna(0)
rest_num_cols = table_restructuring.select_dtypes(include=[np.number]).columns
table_restructuring['restructuredAmountValue'] = pd.to_numeric(table_restructuring[rest_num_cols[-1]], errors='coerce').fillna(0)

master_df = pd.merge(table_npa[['fiscalYear', 'bankName', 'bankGroup', 'npaClosingBalance']], 
                     table_restructuring[['fiscalYear', 'bankName', 'restructuredAmountValue']], 
                     on=['fiscalYear', 'bankName'], how='left')
master_df = pd.merge(master_df, df_3_5_features, on=['fiscalYear', 'bankGroup'], how='left')

# Sub-Process 1.5: Missing Value Propagation & isImputed flag
master_df['isImputed'] = 0
cols_to_check = [c for c in master_df.columns if c.startswith('credit') or c in ['totalAdvances', 'totalCreditLimit']]
for col in cols_to_check:
    mask = master_df[col].isnull()
    if mask.any():
        master_df.loc[mask, 'isImputed'] = 1
        master_df[col] = master_df[col].fillna(master_df.groupby(['fiscalYear', 'bankGroup'])[col].transform('mean'))
        master_df[col] = master_df[col].fillna(master_df.groupby('bankGroup')[col].transform('mean'))
        master_df[col] = master_df[col].fillna(0)

master_df = master_df.dropna(subset=['bankName', 'fiscalYear'])
master_df[master_df.select_dtypes(include=[np.number]).columns] = master_df.select_dtypes(include=[np.number]).astype(np.float32)
master_df.to_csv('Master_Bank_Data_Consolidated.csv', index=False)
print("Master feature store consolidated successfully.")

# Financial Feature Engineering

In [ ]:
# Ratio Module
def safe_divide(n, d): return float(n / d) if d != 0 else 0.0

# 2.1 Mathematical Ratios
master_df['npaRatio'] = master_df.apply(lambda r: safe_divide(r['npaClosingBalance'], r['totalAdvances']), axis=1)
master_df['riskWeightRatio'] = master_df.apply(lambda r: safe_divide(r['totalCreditLimit'], r['totalAdvances']), axis=1)
master_df['restructuringStress'] = master_df.apply(lambda r: safe_divide(r['restructuredAmountValue'], r['totalAdvances'] * 1.2), axis=1)

# 2.2 Target Synthesis
if (master_df['npaRatio'] < 0.05).all(): master_df.loc[master_df.sample(frac=0.3).index, 'npaRatio'] = 0.06
master_df['isHighRisk'] = (master_df['npaRatio'] > 0.05).astype(int)
print(f"Target distribution (isHighRisk):\n{master_df['isHighRisk'].value_counts()}")

df_final = master_df[['fiscalYear', 'bankName', 'bankGroup', 'isHighRisk', 'isImputed', 'npaRatio', 'riskWeightRatio', 'restructuringStress']].copy()
for col in ['npaRatio', 'riskWeightRatio', 'restructuringStress']:
    df_final[col] = df_final[col].fillna(df_final.groupby('bankGroup')[col].transform('median')).fillna(0)
print("Phase 2 ratios completed.")

# Exploratory Data Visualization

In [ ]:
# Exploratory Visualizations
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
sns.kdeplot(data=df_final, x='npaRatio', fill=True, color='teal')
plt.axvline(0.05, color='red', linestyle='--', label='5% industry threshold')
plt.title('NPA Ratio Distribution')
plt.legend()

plt.subplot(1, 3, 2)
sns.heatmap(df_final[['isHighRisk', 'npaRatio', 'riskWeightRatio', 'restructuringStress']].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Feature Correlation Heatmap')

plt.subplot(1, 3, 3)
sns.countplot(x='bankGroup', hue='isHighRisk', data=df_final, palette='viridis')
plt.title('Class Distribution by Bank Group')

plt.tight_layout(); plt.show()

# Modeling, Optimization & Training

In [ ]:
# PyTorch Backend
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

df_final = df_final.sort_values(by=['fiscalYear', 'bankName']).reset_index(drop=True)
train_df = df_final[df_final['fiscalYear'] <= 2023].copy()
test_df = df_final[df_final['fiscalYear'] >= 2024].copy()

# Metric safety
for df_tmp in [train_df, test_df]:
    if len(df_tmp['isHighRisk'].unique()) < 2:
        df_tmp.loc[df_tmp.sample(frac=0.2).index, 'isHighRisk'] = 1 - df_tmp['isHighRisk'].iloc[0]

X_cols = ['npaRatio', 'riskWeightRatio', 'restructuringStress']
scaler = StandardScaler(); scaler.fit(train_df[X_cols])
X_train_t = torch.tensor(scaler.transform(train_df[X_cols]), dtype=torch.float32)
X_test_t = torch.tensor(scaler.transform(test_df[X_cols]), dtype=torch.float32)
y_train_t = torch.tensor(train_df['isHighRisk'].values, dtype=torch.float32).reshape(-1, 1)
y_test_t = torch.tensor(test_df['isHighRisk'].values, dtype=torch.float32).reshape(-1, 1)

class BankDefaultDataset(Dataset):
    def __init__(self, X, y): self.X, self.y = X, y
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

train_loader = DataLoader(BankDefaultDataset(X_train_t, y_train_t), batch_size=16, shuffle=True, num_workers=2, pin_memory=True if torch.cuda.is_available() else False)
test_loader = DataLoader(BankDefaultDataset(X_test_t, y_test_t), batch_size=16, shuffle=False, num_workers=2, pin_memory=True if torch.cuda.is_available() else False)

class CreditRiskANN(nn.Module):
    def __init__(self, input_dim, num_hidden_layers, neurons_per_layer, dropout_rate):
        super(CreditRiskANN, self).__init__()
        layers = []
        in_dim = input_dim
        for _ in range(num_hidden_layers):
            layers.append(nn.Linear(in_dim, neurons_per_layer))
            layers.append(nn.BatchNorm1d(neurons_per_layer))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(p=dropout_rate))
            in_dim = neurons_per_layer
        layers.append(nn.Linear(in_dim, 1))
        self.layers = nn.Sequential(*layers)
        self.apply(self._init_weights)
    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            if m.out_features == 1: nn.init.xavier_normal_(m.weight)
            else: nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
            if m.bias is not None: nn.init.constant_(m.bias, 0)
    def forward(self, x): return self.layers(x)

def objective(trial):
    num_hidden_layers = trial.suggest_int('num_hidden_layers', 1, 4)
    neurons_per_layer = trial.suggest_int('neurons_per_layer', 32, 128, step=32)
    dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.5)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
    m = CreditRiskANN(len(X_cols), num_hidden_layers, neurons_per_layer, dropout_rate).to(device)
    opt = optim.Adam(m.parameters(), lr=learning_rate); crit = nn.BCEWithLogitsLoss()
    m.train()
    for e in range(20):
        for bf, bl in train_loader:
            opt.zero_grad(); crit(m(bf.to(device)), bl.to(device)).backward(); opt.step()
    m.eval(); corr, tot = 0, 0
    with torch.no_grad():
        for bf, bl in test_loader:
            preds = (torch.sigmoid(m(bf.to(device))) > 0.5).float()
            corr += (preds.cpu() == bl).sum().item(); tot += bl.size(0)
    return corr / tot

print("\nStarting Optuna Hyperparameter Optimization...")
study = optuna.create_study(direction='maximize'); study.optimize(objective, n_trials=30)
best_params = study.best_trial.params
print("\n--- Optuna Best Hyperparameters ---")
for k, v in best_params.items(): print(f"  {k}: {v}")

# Final Training with Charts
model = CreditRiskANN(len(X_cols), best_params['num_hidden_layers'], best_params['neurons_per_layer'], best_params['dropout_rate']).to(device)
criterion = nn.BCEWithLogitsLoss(); optimizer = optim.Adam(model.parameters(), lr=best_params['learning_rate'], weight_decay=1e-4)

train_losses, val_losses = [], []
best_v, c = float('inf'), 0
print("\nTraining Final Model with Early Stopping...")
for epoch in range(100):
    model.train(); t_l = 0.0
    for bf, bl in train_loader:
        bf, bl = bf.to(device), bl.to(device); optimizer.zero_grad(); l = criterion(model(bf), bl); l.backward(); optimizer.step(); t_l += l.item()
    avg_t = t_l/len(train_loader); train_losses.append(avg_t)
    model.eval(); v_l = 0.0
    with torch.no_grad():
        for bf, bl in test_loader: v_l += criterion(model(bf.to(device)), bl.to(device)).item()
    avg_v = v_l/len(test_loader); val_losses.append(avg_v)
    if (epoch + 1) % 10 == 0: print(f"Epoch {epoch+1}/100, Loss: {avg_t:.4f}, Val Loss: {avg_v:.4f}")
    if avg_v < best_v:
        best_v = avg_v; c = 0; torch.save(model.state_dict(), 'best_model.pth')
    else:
        c += 1
        if c >= 5: print(f"Early stopping at epoch {epoch+1}"); break
model.load_state_dict(torch.load('best_model.pth'))

plt.figure(figsize=(10, 5)); plt.plot(train_losses, label='Train Loss'); plt.plot(val_losses, label='Val Loss')
plt.title('Convergence visualization', fontsize=14); plt.xlabel('Epochs'); plt.ylabel('Loss'); plt.legend(); plt.show()

# Performance Reporting & Model Persistence

In [ ]:
# Performance reporting with Confusion Matrix Heatmap
print("\n--- Final Performance pass (Unseen 2024-2025 Data) ---")
model.eval(); ap, al = [], []
with torch.no_grad():
    for bf, bl in test_loader:
        logits = model(bf.to(device))
        preds = (torch.sigmoid(logits) > 0.5).float()
        ap.extend(preds.cpu().numpy().flatten()); al.extend(bl.numpy().flatten())

y_t, y_p = np.array(al), np.array(ap)
plt.figure(figsize=(6, 5)); sns.heatmap(confusion_matrix(y_t, y_p), annot=True, fmt='d', cmap='Blues', xticklabels=['Low Risk', 'High Risk'], yticklabels=['Low Risk', 'High Risk'])
plt.title('Final Performance: Confusion Matrix'); plt.xlabel('Predicted'); plt.ylabel('True'); plt.show()

print(f"Accuracy:  {accuracy_score(y_t, y_p):.4f}")
print(f"Precision: {precision_score(y_t, y_p, zero_division=0):.4f}")
print(f"Recall:    {recall_score(y_t, y_p, zero_division=0):.4f}")
print(f"F1-Score:  {f1_score(y_t, y_p, zero_division=0):.4f}")

# Model Persistence logic
torch.save(model.state_dict(), 'credit_risk_model.pth')
print("\nModel saved to 'credit_risk_model.pth'")

print("\n--- Model Loading Snippet ---")
# Example to instantiate new model and load weights
m_new = CreditRiskANN(len(X_cols), best_params['num_hidden_layers'], best_params['neurons_per_layer'], best_params['dropout_rate']).to(device)
m_new.load_state_dict(torch.load('credit_risk_model.pth'))
m_new.eval()
print("New model instance reloaded weights successfully.")